# 02 · Target & Imbalance Analysis

**Amaç:** Class imbalance'ı görselleştirmek ve metrik seçim gerekçesini kurmak.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_style('whitegrid')

from src.data.loader import load_validated


In [ ]:
df, _ = load_validated()
rate = df['IsFraudTransaction'].mean()
print(f'Fraud rate: {rate*100:.4f}%  (1:{int(1/rate)})')
naive_acc = 1 - rate
print(f'"Hep 0 söyle" accuracy: {naive_acc*100:.4f}%  →  accuracy yanıltıcı')

## Kırılım fraud oranları (lift tablosu)

In [ ]:
def rate_tbl(col):
    g = df.groupby(col, dropna=False).agg(n=('IsFraudTransaction','size'), fraud=('IsFraudTransaction','sum'))
    g['rate_pct'] = (g['fraud']/g['n']*100).round(3)
    g['lift'] = (g['rate_pct']/(rate*100)).round(2)
    return g.sort_values('n', ascending=False)

for c in ['TransactionType','DeviceOSName','HasMobileActivationL1H','HasMobileActivationL8H','IsFractionalAmount','CustomerSegment']:
    print('---', c, '---')
    print(rate_tbl(c).to_string())
    print()

## Fraud vs non-fraud — sayısal dağılım karşılaştırmaları

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, ['TransactionAmount','CustomerAge','CustomerTenure','UniqueIPCount']):
    for v, label in [(0,'non-fraud'),(1,'fraud')]:
        sub = df.loc[df['IsFraudTransaction']==v, col]
        if col=='TransactionAmount':
            sub = np.log1p(sub)
        sns.kdeplot(sub, ax=ax, label=label, fill=True, alpha=0.3)
    ax.set_title(col + (' (log1p)' if col=='TransactionAmount' else ''))
    ax.legend()
plt.tight_layout()

## Sonuç
- Recall ve precision@k anlamlı metrikler; accuracy hiçbir karar üretmez.
- HasMobileActivationL1H tek başına 17x lift sağlıyor.
- Amount dağılımları büyük overlap'a sahip — tek başına ayırt edici değil.